Here is how it works:
- Full Clone and extract all config files: .yml, .yaml and related .json, .sh
- Shallow Clone from all branches: afect number commits & contributors build script or config will be only from the latest snapshop

- it sorts and index the url also save the list of random sample url indeces
- after each puase or interuption the "Cloned Repo" should be emptyied
- by a new new rerun it continues the review from the last reviewed url which is log is stored in .evn by START_NUMBER
- the sample repos are stored in "Cloned_Sample"
- this will save the sample repos as well as metrics, configs, builds and test lines
- Full Clone helps to extract full contributors and commit history
- saving metadata happend immediately so it will get lost by pause/start



In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import glob
import random
from pathlib import Path
from dotenv import load_dotenv, set_key
import requests
import time,datetime

# === CONFIGURATION ===
MAX_PROJECTS = 1976
RANDOM_SEED = 42
NUM_SAMPLES_TO_KEEP = 150
ENV_FILE = 'All_tokens.env'

# === LOAD .env ===
load_dotenv(ENV_FILE)
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

START_NUMBER = int(os.getenv("START_NUMBER"))
SAMPLE_LIST_RAW = os.getenv("SAMPLE_LIST", "").strip()

# === PATHS ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
base_dir = Path(r"E:\Android Mobile Project\AndroidProjects")
clone_dir = base_dir / "Cloned repos"
cloned_sample_dir = base_dir / "Cloned_Sample"
yml_output_dir = base_dir / "Config Files"
commits_dir = base_dir / "Commits"
build_info_dir = base_dir / "BuildInfo"
metadata_path = base_dir / "8.2-Project_Metadata.csv"
config_location_csv = base_dir / "Config_Location.csv"

# === ENSURE ALL FOLDERS EXIST ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir, cloned_sample_dir]:
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]
df = df.sort_values(by='github_url').reset_index(drop=True)
df[['github_url']].to_csv(base_dir / 'Sorted_URL_List.csv', index_label='Index')

# === HANDLE SAMPLE_LIST ===
if SAMPLE_LIST_RAW:
    sample_indices_to_keep = set(map(int, SAMPLE_LIST_RAW.split(',')))
    print(f"🔁 Loaded SAMPLE_LIST from .env with {len(sample_indices_to_keep)} indices.")
else:
    random.seed(RANDOM_SEED)
    sample_indices_to_keep = set(random.sample(range(len(df)), min(NUM_SAMPLES_TO_KEEP, len(df))))
    sample_string = ",".join(map(str, sorted(sample_indices_to_keep)))
    set_key(ENV_FILE, 'SAMPLE_LIST', sample_string)
    print(f"🎲 Generated and saved new SAMPLE_LIST with {len(sample_indices_to_keep)} indices.")

# === LOAD EXISTING CONFIG LOCATIONS IF RESUMING ===
if config_location_csv.exists():
    config_locations_df = pd.read_csv(config_location_csv)
else:
    config_locations_df = pd.DataFrame(columns=["repo_name", "config_file_path", "file_type"])

# === HELPER: Get GitHub API count
def get_count(endpoint_url, headers):
    try:
        r = requests.get(endpoint_url, headers=headers, timeout=30)
        if 'Link' in r.headers:
            return int(r.headers['Link'].split('page=')[-1].split('>')[0])
        elif r.ok:
            return len(r.json())
        return 0
    except:
        return 0

# === PROCESS EACH REPO ===
for i in range(START_NUMBER - 1, len(df)):
    url = df.loc[i, 'github_url']
    parts = url.split('/')
    if len(parts) < 5:
        continue
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_index = str(i).zfill(4)
    repo_name = f"{repo_index}.{username}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔍 [{i+1}/{len(df)}] Processing {repo_name}...")

    try:
        subprocess.run(['git', 'clone', url, str(repo_path)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) # full clone
        #subprocess.run(['git', 'clone', '--depth=1', '--no-single-branch', url, str(repo_path)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) # shallow clone from all branches

        print("✅ Clone complete")
    except Exception as e:
        print(f"❌ Clone failed for {repo_name}: {e}")
        continue

    # === Locate CI/CD config files (.yml, .yaml, .json, .sh with CI content) ===
    ci_keywords = ['ci', 'build', 'test', 'workflow', 'pipeline', 'instrumentation']
    config_files_found = []

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_lower = file.lower()
            file_path = Path(root) / file
            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")

            # Always include .yml/.yaml files
            if file_lower.endswith(('.yml', '.yaml')):
                config_files_found.append({
                    "repo_name": repo_name,
                    "config_file_path": rel_path,
                    "file_type": file_lower.split('.')[-1]
                })

            # Conditionally include .json and .sh files based on CI-related keywords
            elif file_lower.endswith(('.json', '.sh')):
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read().lower()
                        if any(keyword in content for keyword in ci_keywords):
                            config_files_found.append({
                                "repo_name": repo_name,
                                "config_file_path": rel_path,
                                "file_type": file_lower.split('.')[-1]
                            })
                except Exception as e:
                    print(f"⚠️ Could not read {rel_path} in {repo_name}: {e}")

    # === Save config file paths ===
    if config_files_found:
        config_locations_df = pd.concat([config_locations_df, pd.DataFrame(config_files_found)], ignore_index=True)

    # === Extract commit history and save it ===
    try:
        commits_output_path = commits_dir / f"{repo_name}_commits.txt"
        with open(commits_output_path, 'w', encoding='utf-8') as f:
            subprocess.run(
                ['git', 'log', '--pretty=format:%h | %an | %ad | %s', '--date=iso'],
                cwd=repo_path,
                stdout=f,
                stderr=subprocess.DEVNULL
            )
        print("📜 Commit history saved")
    except Exception as e:
        print(f"⚠️ Failed to extract commits for {repo_name}: {e}")

        # === Extract contributors and save ===
    try:
        contributors_url = f"{base_api}/contributors"
        contributors_output_path = commits_dir / f"{repo_name}_contributors.csv"
        headers = {'Authorization': f'token {GITHUB_TOKEN}'}
        r = requests.get(contributors_url, headers=headers, timeout=30)

        if r.status_code == 403 and r.headers.get('X-RateLimit-Remaining') == '0':
            reset_ts = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
            reset_time = datetime.datetime.fromtimestamp(reset_ts)
            wait_sec = max(reset_ts - time.time(), 0)
            print(f"⏳ Rate limit hit (contributors). Waiting until {reset_time.strftime('%Y-%m-%d %H:%M:%S')} ({int(wait_sec)}s)...")
            time.sleep(wait_sec + 5)
            r = requests.get(contributors_url, headers=headers, timeout=30)

        if r.ok:
            contrib_data = r.json()
            contrib_rows = [{
                'login': c.get('login'),
                'name': c.get('name', ''),  # GitHub may not expose name in contributor API
                'contributions': c.get('contributions'),
                'profile_url': c.get('html_url')
            } for c in contrib_data]

            pd.DataFrame(contrib_rows).to_csv(contributors_output_path, index=False)
            print("👥 Contributor list saved")
        else:
            print(f"⚠️ Failed to fetch contributors for {repo_name}. Status code: {r.status_code}")

    except Exception as e:
        print(f"⚠️ Contributor extraction failed for {repo_name}: {e}")


    # === Append Metadata Info ===
    try:
        headers = {'Authorization': f'token {GITHUB_TOKEN}'}
        base_api = f"https://api.github.com/repos/{username}/{project}"

        # === Metadata API call with rate limit handling ===
        while True:
            r = requests.get(base_api, headers=headers, timeout=30)

            if r.status_code == 403 and r.headers.get('X-RateLimit-Remaining') == '0':
                reset_ts = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
                reset_time = datetime.datetime.fromtimestamp(reset_ts)
                wait_sec = max(reset_ts - time.time(), 0)
                print(f"⏳ Rate limit hit. Waiting until {reset_time.strftime('%Y-%m-%d %H:%M:%S')} ({int(wait_sec)}s)...")
                time.sleep(wait_sec + 5)  # Wait a bit extra to be safe
                continue  # Retry after sleeping

            if r.ok:
                break  # Exit loop if request is successful

        data = r.json()
        project_metadata = {
            'project_name': project,
            'repo_name': repo_name,
            'full_name': data.get('full_name'),
            'description': data.get('description'),
            'language': data.get('language'),
            'license': data.get('license', {}).get('name') if data.get('license') else None,
            'created_at': data.get('created_at'),
            'updated_at': data.get('updated_at'),
            'last_commit_date': data.get('pushed_at'),
            'stars': data.get('stargazers_count'),
            'forks': data.get('forks_count'),
            'watchers': data.get('watchers_count'),
            'open_issues': data.get('open_issues_count'),
            'contributors': get_count(f"{base_api}/contributors", headers),
            'pull_requests': get_count(f"{base_api}/pulls?state=all", headers),
            'commits': get_count(f"{base_api}/commits", headers),
            'size': data.get('size')
        }

        metadata_df = pd.DataFrame([project_metadata])
        if metadata_path.exists():
            metadata_df.to_csv(metadata_path, mode='a', header=False, index=False)
        else:
            metadata_df.to_csv(metadata_path, mode='w', header=True, index=False)

        print("📊 Metadata saved")


    except Exception as e:
        print(f"⚠️ Metadata error for {repo_name}: {e}")

    # === Move to Cloned_Sample or delete ===
    try:
        if i in sample_indices_to_keep:
            dest_path = cloned_sample_dir / repo_path.name
            if dest_path.exists():
                shutil.rmtree(dest_path, ignore_errors=True)
            shutil.move(str(repo_path), str(dest_path))
            print(f"📦 Sample repo moved to: {dest_path}")
        else:
            shutil.rmtree(repo_path, ignore_errors=True)
            print(f"🗑️ Non-sample repo deleted: {repo_path.name}")
    except Exception as e:
        print(f"❌ Error handling repo folder for {repo_name}: {e}")

    # === Update START_NUMBER ===
    set_key(ENV_FILE, 'START_NUMBER', str(i + 2))

    # === Save Config_Location.csv incrementally to allow stop/resume ===
    config_locations_df.drop_duplicates().to_csv(config_location_csv, index=False)

print(f"\n✅ Process complete. Sampled: {len(sample_indices_to_keep)} | Total Processed: {len(df) - (START_NUMBER - 1)}")
print("\n✅ All selected repositories have been processed.")
